# 01 - Exploratory Data Analysis: Heart Disease UCI (Cleveland)

**Goal:** understand the data before modelling. Run each cell with Shift+Enter, then write your own observations in the *Observations* cell below each output.

Input: `../data/processed/heart_disease_clean.csv` (created by `python -m src.data.preprocess`)
Figures are saved to `../reports/figures/`.

## 1. Setup and load data

In [ ]:
import sys; sys.path.append("..")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", palette="Set2")
FIG = Path("../reports/figures"); FIG.mkdir(parents=True, exist_ok=True)

df = pd.read_csv("../data/processed/heart_disease_clean.csv")
df.head()

## 2. Structure and statistics

In [ ]:
print(df.shape)
df.info()
df.describe().T

**Observations (write your own):**
- Rows x columns:
- Columns with missing values:
- Scale differences between features (age vs cholesterol, etc.):

## 3. Missing values

In [ ]:
missing = df.isna().sum()
print(missing[missing > 0])
print(f"Rows with any missing: {df.isna().any(axis=1).sum()} of {len(df)}")

**Observations (write your own):**
- Which columns are missing and how many:
- Why I will impute inside the training pipeline instead of dropping rows (avoid data leakage, dataset is small):

## 4. Class balance

In [ ]:
ax = sns.countplot(x="target", data=df)
ax.set_xticks([0, 1])
ax.set_xticklabels(["No disease (0)", "Disease (1)"])
ax.set_title("Class Balance")
for p in ax.patches:
    ax.annotate(int(p.get_height()), (p.get_x() + 0.35, p.get_height() + 1))
plt.savefig(FIG / "class_balance.png", dpi=150, bbox_inches="tight")
plt.show()
print(df["target"].value_counts(normalize=True))

**Observations (write your own):**
- Class ratio:
- Is accuracy meaningful? Why I will also track precision, recall and ROC-AUC:

## 5. Distribution of numerical features

In [ ]:
num_cols = ["age", "trestbps", "chol", "thalach", "oldpeak"]
df[num_cols].hist(bins=20, figsize=(14, 8), edgecolor="black")
plt.suptitle("Distribution of Numerical Features", y=1.02)
plt.tight_layout()
plt.savefig(FIG / "histograms.png", dpi=150, bbox_inches="tight")
plt.show()

**Observations (write your own):**
- Shape of each feature (bell-shaped, skewed, long tail):
- Which features are skewed (e.g. `oldpeak`):

## 6. Numerical features split by target

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.ravel(), num_cols):
    sns.histplot(data=df, x=col, hue="target", kde=True, ax=ax, bins=20)
axes.ravel()[-1].axis("off")
plt.tight_layout()
plt.savefig(FIG / "histograms_by_target.png", dpi=150)
plt.show()

**Observations (write your own):**
- Features where sick and healthy separate (`thalach`, `oldpeak`, `age`?):
- Features where they overlap heavily (`chol`, `trestbps`?):

## 7. Boxplots (outliers)

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, col in zip(axes, num_cols):
    sns.boxplot(data=df, x="target", y=col, ax=ax)
plt.tight_layout()
plt.savefig(FIG / "boxplots.png", dpi=150)
plt.show()

**Observations (write your own):**
- Where outliers appear:
- Decision: keep them because they are clinically plausible and the dataset is small; scaling reduces their influence.

## 8. Categorical features vs target

In [ ]:
cat_cols = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
for ax, col in zip(axes.ravel(), cat_cols):
    sns.countplot(data=df, x=col, hue="target", ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.savefig(FIG / "categorical_vs_target.png", dpi=150)
plt.show()

**Observations (write your own):**
- Strongest categorical signals (`cp`, `exang`, `thal`, `ca`, `sex`?):
- Weak signals (`fbs`?):

## 9. Correlation heatmap

In [ ]:
plt.figure(figsize=(11, 8))
corr = df.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.savefig(FIG / "correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

print(corr["target"].sort_values(ascending=False))

**Observations (write your own):**
- Top positive and negative correlations with `target`:
- Any feature pair above 0.8 (redundancy)?
- Caveat: correlation shows only linear relationships and does not prove causation.

## 10. Summary of findings (write 5-8 bullets in your own words)
- Dataset size, number of features, missing cells:
- Class balance:
- Most predictive features:
- Weak features:
- Scaling needed because:
- One-hot encoding needed because:
- Outlier decision:
- Redundancy check result:

**Next step:** build the preprocessing pipeline (impute, scale, encode) and train models.